! Hier ist erst einmal die Idee, wie man das mit Clustern machen könnte. Muss dementsprechend noch angepasst werden, wenn die finalen Cluster stehen

In [ ]:
#!pip install mistralAI

Import libraries

In [14]:
import os
import json
import random
import time
import re
import matplotlib.pyplot as plt
import networkx as nx
from mistralai import Mistral
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
from collections import defaultdict

Set up API

In [15]:
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
model =  "open-mistral-nemo"
client = Mistral(api_key=api_key)

Set file paths and load data

In [ ]:
#File paths
#input_path = r"filtered_output.json"
cq_path = r"competency_questions_output/competency_questions_all_documents.txt"
output_file_path = "ontology_output/ontology_cluster_based.json"
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

#Load data
with open(input_path, "r", encoding="utf-8") as f:
    documents = json.load(f)
    
#Load competency questions
cq_by_doc_id = {}
with open(cq_path, "r", encoding="utf-8") as f:
    content = f.read()

cq_blocks = re.split(r'Dokument:\s*', content)
for block in cq_blocks[1:]:
    lines = block.strip().splitlines()
    doc_id = lines[0].strip()
    frage = next((l for l in lines if l.lower().startswith("frage:")), None)
    quelle = next((l for l in lines if "quelle" in l.lower()), None)
    if frage and quelle:
        cq_by_doc_id[doc_id] = {
            "question": frage.replace("Frage:", "").strip(),
            "source": quelle.replace("Quelle:", "").strip()
        }



Cluster the questions using TF-IDF + Ward

In [ ]:
# Load & vectorize questions
questions = [entry["question"] for entry in cq_by_doc_id.values() if "question" in entry and entry["question"].strip()]
if not questions:
    raise ValueError "No valid questions found in 'cq_by_doc_id'. Ensure that 'cq_by_doc_id' is populated correctly."q
estions:
    raise ValueError("No valid questions found in 'cq_by_doc_id'. Ensure that 'cq_by_doc_id' is populated correctly.")

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(questions)
()]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(questions)
    raise ValueError("No valid questions found in 'cq_by_doc_id'. Ensure that 'cq_by_doc_id' is populated correctly.")

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(questions)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(questions)
Z = linkage(distances, method="ward")

# Decide number of clusters
num_clusters = 5  # tune this as needed
cluster_labels = fcluster(Z, num_clusters, criterion="maxclust")

# Extract group ID's by cluster
doc_ids = list(cq_by_doc_id.keys())
doc_cluster_map = {doc_id: cluster_labels[i] for i, doc_id in enumerate(doc_ids)}

cluster_docs = defaultdict(list)
for doc_id, cluster_id in doc_cluster_map.items():
    if doc_id in documents:
        cluster_docs[cluster_id].append(doc_id)


ValueError: empty vocabulary; perhaps the documents only contain stop words

Ontology system prompt

In [19]:
#Ontology system prompt
ontology_system_prompt = """
You are an expert legal ontology engineer.

Your task is to extract ontology components from the given legal ruling text (German court decisions). 
Identify and return three structured components:

1. **Classes**: Legal entities or concepts (e.g., Urteil, Entscheidungsgründe, Anspruch, Person, Tatbestand, Eiwendung, Rubrum, Tenor)
2. **Properties**: Verbs or phrases representing relationships or attributes (e.g., beinhaltet, basiert auf, verhindert, erörtern)
3. **Relationships**: Triples (Subject)-(Predicate)-(Object), connecting two classes using a property

Requirements:
- Only extract concepts if they have relationships.
- Use clear legal language in German.
- Output must follow this structure (example):

{
  "classes": ["Urteil", "Anspruch", "Tatbestand"],
  "properties": ["beinhaltet", "basiert auf"],
  "relationships": [
    ["Urteil", "beinhaltet", "Tatbestand"],
    ["Anspruch", "basiert auf", "Tatbestand"]
  ]
}

## Few-Shot Examples 
"<Example 1>"
"Sentence: 'Ein Arbeitnehmer hat eine Sozialversicherungsnummer'"
"Classes: [Arbeitnehmner, Sozialversicherungsnummer]"
"Properties: [hatIdentifikation]"
"Relationship: [(Arbeitnehmer)-(hatIdentifikation)-(Sozialversicherungsnummer)]"

"<Example 2>"
"Sentence: 'Arbeitnehmer haben nach Gesetz einen geregelten Mindestanspruch auf 24 Tage Urlaub im Jahr'"
"Classes: [Arbeitnehmer, Urlaub]"
"Properties: [hatMindestAnspruchAuf]"
"Relationship: [(Arbeitnehmer)-(hatMindestAnspruchAuf)-(Urlaub)]"

## Modelling Guidelines
"- **Concept Identification**:"
"- Identify nouns and noun phrases as potential **Classes**."
"- Identify verbs and verb phrases as potential **Properties**."
"- Identify prepositions that establish relationships between nouns."

"- **Classes**:"
"- Represent concepts in the domain, not the words that denote these concepts."
"- Avoid creating classes for synonyms; use a single class for concepts with the same meaning."

"- **Properties**:"
"- Represent significant relationships or attributes between classes."
"- Should be meaningful and represent a significant connection."

"- **Relationships**:"
"- Establish connections between classes using properties."
"- Ensure relationships are meaningful within the domain context."

"- **General Principles**:"
"- There is no single correct way to model a domain; the best solution depends on the application."
"- Ontology development is an iterative process; refine as needed."
"- Avoid cycles in the class hierarchy."
"- Siblings in the hierarchy should be at the same level of generality."

## Naming Conventions
"**General**:"
"- Do not add strings like 'class', 'domain', 'range', 'property', or 'slot' to names."
"- Use consistent naming throughout the ontology."

"**Classes**:"
"- Names are always capitalized."
"- Use nouns or compound nouns (e.g., Vertrag, Arbeit, Arbeitsvertrag)."
"- Use singular over plural (e.g., Arbeitsvertrag instead of Arbeitsverträge)."
"- Avoid abbreviations (e.g., Arbeitgeber instead of AG)."

"**Properties**:"
"- Names start with a lower-case letter."
"- Use verbs or verb phrases."
"- Can contain nouns in CamelCase starting with a verb (e.g., hatAnspruchAuf)."
"- Do not include spaces, commas, asterisks, or special characters."

##Finally
"Ensure that you include empty lists for classes, properties, or relationships if none are found."
"Make sure all extracted components, i.e. classes, properties, and relationships as well as all descriptions are in German and translate where necessary."
"Include a class only if at least one relationship is found for a class. Verify this requirement."
"Check carefully for each class without any relationship, based on the name and the description of the class, if it can be merged with another class or if it is actually a relationship between on class and another. This is particularly relevant for classes that are named in the form of 'has something'."
"All ontology components must be in German language!"
"Terms like 'range' or 'domain' are never allowed!"

Output must be valid JSON with no comments or explanation.
"""

#Prompt Builder 
def build_cluster_ontology_prompt(texts, questions):
    joined_text = "\n\n".join(texts)
    joined_questions = "\n- ".join(questions)
    return f"""Extract ontology components from the following German legal texts. 
The extraction should be informed by these competency questions:

- {joined_questions}

Texts:
{joined_text[:12000]}
"""


Cluste-based ontology extractiom

In [20]:
ontology_results = {}
for cluster_id, doc_ids in cluster_docs.items():
    cluster_texts = []
    cluster_questions = []
    for doc_id in doc_ids:
        text_data = documents[doc_id].get("text", {}).get("entscheidungsinhalt", {})
        gruende_list = text_data.get("gruende", {}).get("gruende", [])
        if not gruende_list:
            continue
        cluster_texts.append("\n".join(gruende_list))
        cluster_questions.append(cq_by_doc_id[doc_id]["question"])

    if not cluster_texts:
        continue

    prompt = build_cluster_ontology_prompt(cluster_texts, cluster_questions)

    try:
        response = client.chat.complete(
            model=model,
            messages=[
                {"role": "system", "content": ontology_system_prompt},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5
        )
        content = response.choices[0].message.content.strip()
        try:
            ontology = json.loads(content)
            ontology_results[cluster_id] = {
                "ontology": ontology,
                "questions": cluster_questions,
                "doc_ids": doc_ids
            }
            print(f"✅ Ontology extracted for cluster {cluster_id}")
        except json.JSONDecodeError:
            print(f"❌ Invalid JSON for cluster {cluster_id}")
        time.sleep(1)
    except Exception as e:
        print(f"❌ Error for cluster {cluster_id}: {e}")

# === Save Results ===
with open(output_file_path, "w", encoding="utf-8") as f_out:
    json.dump(ontology_results, f_out, indent=2, ensure_ascii=False)

print("✅ Cluster-based ontology extraction completed.")


NameError: name 'cluster_docs' is not defined